In [1]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from google import genai
from banking77_llm_vs_ml.core import (
    load_data, make_schema, build_few_shot_prompt, run_eval, PROMPT_V1,
)

client = genai.Client()
MODEL = "gemini-3.5-flash-lite"

train_df, val_df, test = load_data()
categories = sorted(train_df["category"].unique()) + ["out_of_scope"]
schema = make_schema(categories)
sample = val_df.groupby("category").sample(3, random_state=42).reset_index(drop=True)

In [2]:
PROMPT_V2 = build_few_shot_prompt(train_df, per_category=3)
print(PROMPT_V2[:1500])
print("...\nTotal characters:", len(PROMPT_V2))

You classify customer messages sent to a bank's support team.
Choose the single best category for the message.
If the message is not about banking, choose out_of_scope.

Real example messages for each category:
- Refund_not_showing_up: Why hasn't my refund been appiled to my account? | I requested a refund but am unable to see it in my account. Why is that? | My refund isn't showing up on my statement.
- activate_my_card: I need to activate my card, how is that done? | Who do I call to activate my new card? | Card activation is not working. What do i do?
- age_limit: How old does my daughter have to be to open an account? | What is minimum age for an account? | what is the age limit for opening a new account
- apple_pay_or_google_pay: Can I deposit money using Apple Pay? | Am I able to do top-ups with Google Pay? | Why isn't my Google Pay top up working for me? Did I do something wrong?
- atm_support: What is the closest ATM? | Will any ATM's decline my card? | Which locations can I wi

In [3]:
v2 = run_eval(client, MODEL, PROMPT_V2, schema, sample["text"], "results/v2_few_shot_flash_lite.csv")

176/231

ReadError: [WinError 10053] An established connection was aborted by the software in your host machine

In [4]:
v2 = run_eval(client, MODEL, PROMPT_V2, schema, sample["text"], "results/v2_few_shot_flash_lite.csv")

231/231
Done: 231 results in results\v2_few_shot_flash_lite.csv


In [5]:
import time
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

p1 = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, C=10))
p1.fit(train_df["text"], train_df["category"])
sample["p1_pred"] = p1.predict(sample["text"])

def summarize(name, preds, tokens_in=None, latency_s=None):
    return {
        "model": name,
        "accuracy": round(accuracy_score(sample["category"], preds), 3),
        "macro_f1": round(f1_score(sample["category"], preds, average="macro", zero_division=0), 3),
        "avg_tokens_in": None if tokens_in is None else round(tokens_in.mean()),
        "p50_latency_s": None if latency_s is None else latency_s.quantile(0.5),
    }

rows = [summarize("P1: TF-IDF + LogReg", sample["p1_pred"])]
for name, path in [("LLM v1: zero-shot", "results/v1_flash_lite.csv"),
                   ("LLM v2: few-shot (3/category)", "results/v2_few_shot_flash_lite.csv")]:
    r = sample.merge(pd.read_csv(path), on="text")
    rows.append(summarize(name, r["pred"].fillna("FAILED"), r["tokens_in"], r["latency_s"]))

pd.DataFrame(rows)

,model,accuracy,macro_f1,avg_tokens_in,p50_latency_s
0,P1: TF-IDF + LogReg,0.913,0.911,NaN,NaN
1,LLM v1: zero-shot,0.736,0.714,53.0,1.01
2,LLM v2: few-shot (3/category),0.706,0.713,3915.0,2.83


In [6]:
v2 = run_eval(client, MODEL, PROMPT_V2, schema, sample["text"], "results/v2_few_shot_flash_lite.csv")

231/231
Done: 231 results in results\v2_few_shot_flash_lite.csv


In [9]:
import time
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

p1 = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, C=10))
p1.fit(train_df["text"], train_df["category"])
sample["p1_pred"] = p1.predict(sample["text"])

def summarize(name, preds, tokens_in=None, latency_s=None):
    return {
        "model": name,
        "accuracy": round(accuracy_score(sample["category"], preds), 3),
        "macro_f1": round(f1_score(sample["category"], preds, average="macro", zero_division=0), 3),
        "failed": int((preds == "FAILED").sum()),
        "avg_tokens_in": None if tokens_in is None else round(tokens_in.mean()),
        "p50_latency_s": None if latency_s is None else latency_s.quantile(0.5),
        "p95_latency_s": None if latency_s is None else latency_s.quantile(0.95),
    }
rows = [summarize("P1: TF-IDF + LogReg", sample["p1_pred"])]
for name, path in [("LLM v1: zero-shot", "results/v1_flash_lite.csv"),
                   ("LLM v2: few-shot (3/category)", "results/v2_few_shot_flash_lite.csv")]:
    r = sample.merge(pd.read_csv(path), on="text")
    rows.append(summarize(name, r["pred"].fillna("FAILED"), r["tokens_in"], r["latency_s"]))

summary = pd.DataFrame(rows)
summary

,model,accuracy,macro_f1,failed,avg_tokens_in,p50_latency_s,p95_latency_s
0,P1: TF-IDF + LogReg,0.913,0.911,0,NaN,NaN,NaN
1,LLM v1: zero-shot,0.736,0.714,0,53.0,1.01,3.335
2,LLM v2: few-shot (3/category),0.762,0.746,14,3915.0,2.94,13.566


In [10]:
# P1's confidence for each message (its top probability)
sample["p1_conf"] = p1.predict_proba(sample["text"]).max(axis=1)

def hybrid(llm_path, threshold):
    r = sample.merge(pd.read_csv(llm_path)[["text", "pred"]], on="text", how="left")
    use_llm = (r["p1_conf"] < threshold) & r["pred"].notna()   # P1 unsure AND the LLM answered
    final = r["p1_pred"].where(~use_llm, r["pred"])           # otherwise keep P1's answer
    return final, use_llm.mean()

rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    for name, path in [("v1", "results/v1_flash_lite.csv"),
                       ("v2", "results/v2_few_shot_flash_lite.csv")]:
        final, share = hybrid(path, t)
        rows.append({
            "threshold": t,
            "llm": name,
            "accuracy": round(accuracy_score(sample["category"], final), 3),
            "llm_calls_%": round(share * 100, 1),
        })

hybrid_table = pd.DataFrame(rows).pivot(index="threshold", columns="llm",
                                        values=["accuracy", "llm_calls_%"])
hybrid_table

accuracy        llm_calls_%      
llm             v1     v2          v1    v2
threshold                                  
0.3          0.913  0.922         4.8   4.3
0.4          0.905  0.918         6.9   6.5
0.5          0.883  0.900        12.6  11.7
0.6          0.874  0.905        19.9  19.0
0.7          0.857  0.900        27.7  26.0
0.8          0.818  0.870        40.3  38.1